# 0. Problem
## 1341. Movie Rating — Medium
Return two rows in one column `results`: top reviewer by number of ratings (tie → smallest name), then top February 2020 movie by average rating (tie → smallest title).

Official: https://leetcode.com/problems/movie-rating/

# 1. Setup

In [ ]:
import pandas as pd
movies_rows=[(1,"Avengers"),(2,"Frozen 2"),(3,"Joker")]
users_rows=[(1,"Daniel"),(2,"Monica"),(3,"Maria"),(4,"James")]
ratings_rows=[(1,1,3,"2020-01-12"),(1,2,4,"2020-02-11"),(1,3,2,"2020-02-12"),(1,4,1,"2020-01-01"),(2,1,5,"2020-02-17"),(2,2,2,"2020-02-01"),(2,3,2,"2020-03-01"),(3,1,3,"2020-02-22"),(3,2,4,"2020-02-25")]
movies_pd=pd.DataFrame(movies_rows,columns=["movie_id","title"])
users_pd=pd.DataFrame(users_rows,columns=["user_id","name"])
ratings_pd=pd.DataFrame(ratings_rows,columns=["movie_id","user_id","rating","created_at"])
ratings_pd["created_at"]=pd.to_datetime(ratings_pd["created_at"])

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
movies_spark=spark.createDataFrame(movies_rows,["movie_id","title"])
users_spark=spark.createDataFrame(users_rows,["user_id","name"])
ratings_spark=spark.createDataFrame(ratings_rows,["movie_id","user_id","rating","created_at"]).withColumn("created_at",F.to_date("created_at"))
movies_spark.createOrReplaceTempView("Movies")
users_spark.createOrReplaceTempView("Users")
ratings_spark.createOrReplaceTempView("MovieRating")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
WITH user_counts AS (
  SELECT u.name,COUNT(*) AS cnt
  FROM MovieRating r JOIN Users u ON r.user_id=u.user_id
  GROUP BY u.user_id,u.name
),
feb_movie AS (
  SELECT m.title,AVG(r.rating) AS avg_rating
  FROM MovieRating r JOIN Movies m ON r.movie_id=m.movie_id
  WHERE r.created_at BETWEEN DATE('2020-02-01') AND DATE('2020-02-29')
  GROUP BY m.movie_id,m.title
),
top_user AS (
  SELECT name AS results FROM user_counts ORDER BY cnt DESC,name ASC LIMIT 1
),
top_movie AS (
  SELECT title AS results FROM feb_movie ORDER BY avg_rating DESC,title ASC LIMIT 1
)
SELECT results FROM top_user
UNION ALL
SELECT results FROM top_movie
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
user_counts=(ratings_pd.groupby("user_id",as_index=False).size().rename(columns={"size":"cnt"}).merge(users_pd,on="user_id").sort_values(["cnt","name"],ascending=[False,True]))
top_user=user_counts.iloc[[0]][["name"]].rename(columns={"name":"results"})
feb=ratings_pd.loc[ratings_pd["created_at"].between(pd.Timestamp("2020-02-01"),pd.Timestamp("2020-02-29"))]
movie_avg=(feb.groupby("movie_id",as_index=False).agg(avg_rating=("rating","mean")).merge(movies_pd,on="movie_id").sort_values(["avg_rating","title"],ascending=[False,True]))
top_movie=movie_avg.iloc[[0]][["title"]].rename(columns={"title":"results"})
result_pd=pd.concat([top_user,top_movie],ignore_index=True)
result_pd

# 4. PySpark Solution

In [ ]:
user_counts=(ratings_spark.groupBy("user_id").agg(F.count("*").alias("cnt")).join(users_spark,on="user_id").orderBy(F.desc("cnt"),F.asc("name")))
top_user=user_counts.select(F.col("name").alias("results")).limit(1)
feb=ratings_spark.filter(F.col("created_at").between(F.to_date(F.lit("2020-02-01")),F.to_date(F.lit("2020-02-29"))))
movie_avg=(feb.groupBy("movie_id").agg(F.avg("rating").alias("avg_rating")).join(movies_spark,on="movie_id").orderBy(F.desc("avg_rating"),F.asc("title")))
top_movie=movie_avg.select(F.col("title").alias("results")).limit(1)
result_spark=top_user.unionByName(top_movie)
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| top + tie-break | `ORDER BY metric DESC,name ASC` | `.sort_values()` | `.orderBy()` |
| combine two answers | `UNION ALL` | `pd.concat()` | `.unionByName()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Movies, Users, MovieRating

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: movies_pd, users_pd, ratings_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: movies_spark, users_spark, ratings_spark